# **Laboratorio 12: 🚀 Despliegue 🚀**

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

### **Cuerpo Docente:**

- Profesores: Ignacio Meza, Sebastián Tinoco
- Auxiliar: Eduardo Moya
- Ayudantes: Nicolás Ojeda, Melanie Peña, Valentina Rojas

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Sofia Lazcano
- Nombre de alumno 2: Maria Jesus Espinoza

### **Link de repositorio de GitHub:** https://github.com/jesuow/LabPro

## Temas a tratar

- Entrenamiento y registro de modelos usando MLFlow.
- Despliegue de modelo usando FastAPI
- Containerización del proyecto usando Docker

## Reglas:

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibidas las copias.
- Pueden usar cualquer matrial del curso que estimen conveniente.

### Objetivos principales del laboratorio

- Generar una solución a un problema a partir de ML
- Desplegar su solución usando MLFlow, FastAPI y Docker

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# **Introducción**

<p align="center">
  <img src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExODJnMHJzNzlkNmQweXoyY3ltbnZ2ZDlxY2c0aW5jcHNzeDNtOXBsdCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/AbPdhwsMgjMjax5reo/giphy.gif" width="400">
</p>



Consumida en la tristeza el despido de Renacín, Smapina ha decaído en su desempeño, lo que se ha traducido en un irregular tratamiento del agua. Esto ha implicado una baja en la calidad del agua, llegando a haber algunos puntos de la comuna en la que el vital elemento no es apto para el consumo humano. Es por esto que la sanitaria pública de la municipalidad de Maipú se ha contactado con ustedes para que le entreguen una urgente solución a este problema (a la vez que dejan a Smapina, al igual que Renacín, sin trabajo 😔).

El problema que la empresa le ha solicitado resolver es el de elaborar un sistema que les permita saber si el agua es potable o no. Para esto, la sanitaria les ha proveido una base de datos con la lectura de múltiples sensores IOT colocados en diversas cañerías, conductos y estanques. Estos sensores señalan nueve tipos de mediciones químicas y más una etiqueta elaborada en laboratorio que indica si el agua es potable o no el agua.

La idea final es que puedan, en el caso que el agua no sea potable, dar un aviso inmediato para corregir el problema. Tenga en cuenta que parte del equipo docente vive en Maipú y su intoxicación podría implicar graves problemas para el cierre del curso.

Atributos:

1. pH value
2. Hardness
3. Solids (Total dissolved solids - TDS)
4. Chloramines
5. Sulfate
6. Conductivity
7. Organic_carbon
8. Trihalomethanes
9. Turbidity

Variable a predecir:

10. Potability (1 si es potable, 0 no potable)

Descripción de cada atributo se pueden encontrar en el siguiente link: [dataset](https://www.kaggle.com/adityakadiwal/water-potability)

# **1. Optimización de modelos con Optuna + MLFlow (2.0 puntos)**

El objetivo de esta sección es que ustedes puedan combinar Optuna con MLFlow para poder realizar la optimización de los hiperparámetros de sus modelos.

Como aún no hemos hablado nada sobre `MLFlow` cabe preguntarse: **¡¿Qué !"#@ es `MLflow`?!**

<p align="center">
  <img src="https://media.tenor.com/eusgDKT4smQAAAAC/matthew-perry-chandler-bing.gif" width="400">
</p>

## **MLFlow**

`MLflow` es una plataforma de código abierto que simplifica la gestión y seguimiento de proyectos de aprendizaje automático. Con sus herramientas, los desarrolladores pueden organizar, rastrear y comparar experimentos, además de registrar modelos y controlar versiones.

<p align="center">
  <img src="https://spark.apache.org/images/mlflow-logo.png" width="350">
</p>

Si bien esta plataforma cuenta con un gran número de herramientas y funcionalidades, en este laboratorio trabajaremos con dos:
1. **Runs**: Registro que constituye la información guardada tras la ejecución de un entrenamiento. Cada `run` tiene su propio run_id, el cual sirve como identificador para el entrenamiento en sí mismo. Dentro de cada `run` podremos acceder a información como los hiperparámetros utilizados, las métricas obtenidas, las librerías requeridas y hasta nos permite descargar el modelo entrenado.
2. **Experiments**: Se utilizan para agrupar y organizar diferentes ejecuciones de modelos (`runs`). En ese sentido, un experimento puede agrupar 1 o más `runs`. De esta manera, es posible también registrar métricas, parámetros y archivos (artefactos) asociados a cada experimento.

### **Todo bien pero entonces, ¿cómo se usa en la práctica `MLflow`?**

Es sencillo! Considerando un problema de machine learning genérico, podemos registrar la información relevante del entrenamiento ejecutando `mlflow.autolog()` antes entrenar nuestro modelo. Veamos este bonito ejemplo facilitado por los mismos creadores de `MLflow`:

```python
#!pip install mlflow
import mlflow # importar mlflow

from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

db = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)

mlflow.autolog() # registrar automáticamente información del entrenamiento
with mlflow.start_run(): # delimita inicio y fin del run
    # aquí comienza el run
    rf.fit(X_train, y_train) # train the model
    predictions = rf.predict(X_test) # Use the model to make predictions on the test dataset.
    # aquí termina el run
```

Si ustedes ejecutan el código anterior en sus máquinas locales (desde un jupyter notebook por ejemplo) se darán cuenta que en su directorio *root* se ha creado la carpeta `mlruns`. Esta carpeta lleva el tracking de todos los entrenamientos ejecutados desde el directorio root (importante: si se cambian de directorio y vuelven a ejecutar el código anterior, se creará otra carpeta y no tendrán acceso al entrenamiento anterior). Para visualizar estos entrenamientos, `MLflow` nos facilita hermosa interfaz visual a la que podemos acceder ejecutando:

```
mlflow ui
```

y luego pinchando en la ruta http://127.0.0.1:5000 que nos retorna la terminal. Veamos en vivo algunas de sus funcionalidades!

<p align="center">
  <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZXVuM3A5MW1heDFpa21qbGlwN2pyc2VoNnZsMmRzODZxdnluemo2bCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/3o84sq21TxDH6PyYms/giphy.gif" width="400">
</p>

Les dejamos también algunos comandos útiles:

- `mlflow.create_experiment("nombre_experimento")`: Les permite crear un nuevo experimento para agrupar entrenamientos
- `mlflow.log_metric("nombre_métrica", métrica)`: Les permite registrar una métrica *custom* bajo el nombre de "nombre_métrica"


## **1.1 Combinando Optuna + MLflow (2.0 puntos)**

Ahora que tenemos conocimiento de ambas herramientas, intentemos ahora combinarlas para **más sabor**. El objetivo de este apartado es simple: automatizar la optimización de los parámetros de nuestros modelos usando `Optuna` y registrando de forma automática cada resultado en `MLFlow`.

Considerando el objetivo planteado, se le pide completar la función `optimize_model`, la cual debe:
- **Optimizar los hiperparámetros del modelo `XGBoost` usando `Optuna`.**
- **Registrar cada entrenamiento en un experimento nuevo**, asegurándose de que la métrica `f1-score` se registre como `"valid_f1"`. No se deben guardar todos los experimentos en *Default*; en su lugar, cada `experiment` y `run` deben tener nombres interpretables, reconocibles y diferentes a los nombres por defecto (por ejemplo, para un run: "XGBoost con lr 0.1").
- **Guardar los gráficos de Optuna** dentro de una carpeta de artefactos de Mlflow llamada `/plots`.
- **Devolver el mejor modelo** usando la función `get_best_model` y serializarlo en el disco con `pickle.dump`. Luego, guardar el modelo en la carpeta `/models`.
- **Guardar el código en `optimize.py`**. La ejecución de `python optimize.py` debería ejecutar la función `optimize_model`.
- **Guardar las versiones de las librerías utilizadas** en el desarrollo.
- **Respalde las configuraciones del modelo final y la importancia de las variables** en un gráfico dentro de la carpeta `/plots` creada anteriormente.

*Hint: Le puede ser útil revisar los parámetros que recibe `mlflow.start_run`*

```python
def get_best_model(experiment_id):
    runs = mlflow.search_runs(experiment_id)
    best_model_id = runs.sort_values("metrics.valid_f1")["run_id"].iloc[0]
    best_model = mlflow.sklearn.load_model("runs:/" + best_model_id + "/model")

    return best_model
```

In [ ]:
import optuna
import mlflow
import mlflow.xgboost
import xgboost as xgb
import pickle
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from optuna.visualization import plot_optimization_history, plot_param_importances

#Cargar los datos
def load_data():
    df = pd.read_csv("water_potability.csv")
    df = df.dropna()  
    X = df.drop("Potability", axis=1)
    y = df["Potability"]
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    return train_test_split(X, y, test_size=0.3, random_state=42)

#Objetivo para Optuna
def objective(trial, X_train, X_valid, y_train, y_valid):
    #Hiperparámetros
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "use_label_encoder": False,
        "objective": "binary:logistic",
    }

    #Creación del modelo
    model = xgb.XGBClassifier(**params)

    model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],  
    eval_metric="logloss",         
    early_stopping_rounds=10,      
    verbose=False                  
    )

    # Predicción y métrica
    y_pred = model.predict(X_valid)
    f1 = f1_score(y_valid, y_pred, average="weighted")

    with mlflow.start_run(run_name=f"XGBoost con lr {params['learning_rate']}"):
        mlflow.log_params(params)
        mlflow.log_metric("valid_f1", f1)

    return f1

def optimize_model():
    # Cargar datos
    X_train, X_valid, y_train, y_valid = load_data()


    experiment_name = "Water Potability Optimization"
    mlflow.set_experiment(experiment_name)
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id


    study = optuna.create_study(direction="maximize")
    study.optimize(lambda trial: objective(trial, X_train, X_valid, y_train, y_valid), n_trials=50)

    # gráficos de Optuna
    os.makedirs("plots", exist_ok=True)
    plot_optimization_history(study).write_image("plots/optimization_history.png")
    plot_param_importances(study).write_image("plots/param_importances.png")
    mlflow.log_artifact("plots/optimization_history.png", artifact_path="plots")
    mlflow.log_artifact("plots/param_importances.png", artifact_path="plots")

    #importancia de variables
    best_params = study.best_params
    model = xgb.XGBClassifier(**best_params)
    model.fit(X_train, y_train)
    importance = model.feature_importances_
    importance_df = pd.DataFrame({"Feature": range(len(importance)), "Importance": importance})
    importance_df.to_csv("plots/feature_importance.csv", index=False)
    mlflow.log_artifact("plots/feature_importance.csv", artifact_path="plots")

    os.makedirs("models", exist_ok=True)
    model_path = "models/best_model.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    mlflow.log_artifact(model_path, artifact_path="models")

    return model

def get_best_model(experiment_id):
    runs = mlflow.search_runs(experiment_ids=[experiment_id])
    best_model_id = runs.sort_values("metrics.valid_f1", ascending=False)["run_id"].iloc[0]
    best_model = mlflow.xgboost.load_model(f"runs:/{best_model_id}/model")
    return best_model

if __name__ == "__main__":
    optimize_model()


[I 2024-11-30 23:02:43,447] A new study created in memory with name: no-name-56dca63f-8211-4f4d-ad9a-f14621e13dcb
/home/toti/miniconda3/envs/MDS/lib/python3.12/site-packages/xgboost/sklearn.py:1395: UserWarning: `use_label_encoder` is deprecated in 1.7.0.
  warnings.warn("`use_label_encoder` is deprecated in 1.7.0.")
/home/toti/miniconda3/envs/MDS/lib/python3.12/site-packages/xgboost/sklearn.py:835: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/home/toti/miniconda3/envs/MDS/lib/python3.12/site-packages/xgboost/sklearn.py:835: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
[I 2024-11-30 23:02:43,788] Trial 0 finished with value: 0.6164552694078907 and parameters: {'max_depth': 4, 'learning_rate': 0.089849114814089

# **2. FastAPI (2.0 puntos)**

<div align="center">
  <img src="https://media3.giphy.com/media/YQitE4YNQNahy/giphy-downsized-large.gif" width="500">
</div>

Con el modelo ya entrenado, la idea de esta sección es generar una API REST a la cual se le pueda hacer *requests* para así interactuar con su modelo. En particular, se le pide:

- Guardar el código de esta sección en el archivo `main.py`. Note que ejecutar `python main.py` debería levantar el servidor en el puerto por defecto.
- Defina `GET` con ruta tipo *home* que describa brevemente su modelo, el problema que intenta resolver, su entrada y salida.
- Defina un `POST` a la ruta `/potabilidad/` donde utilice su mejor optimizado para predecir si una medición de agua es o no potable. Por ejemplo, una llamada de esta ruta con un *body*:

```json
{
   "ph":10.316400384553162,
   "Hardness":217.2668424334475,
   "Solids":10676.508475429378,
   "Chloramines":3.445514571005745,
   "Sulfate":397.7549459751925,
   "Conductivity":492.20647361771086,
   "Organic_carbon":12.812732207582542,
   "Trihalomethanes":72.28192021570328,
   "Turbidity":3.4073494284238364
}
```

Su servidor debería retornar una respuesta HTML con código 200 con:


```json
{
  "potabilidad": 0 # respuesta puede variar según el clasificador que entrenen
}
```

**`HINT:` Recuerde que puede utilizar [http://localhost:8000/docs](http://localhost:8000/docs) para hacer un `POST`.**

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib  

# Crear una instancia de la aplicación FastAPI
app = FastAPI()

class WaterQuality(BaseModel):
    ph: float
    Hardness: float
    Solids: float
    Chloramines: float
    Sulfate: float
    Conductivity: float
    Organic_carbon: float
    Trihalomethanes: float
    Turbidity: float

model = joblib.load("models/best_model.pkl")

@app.get("/")
def home():
    return {"model": "Modelo de predicción de potabilidad del agua", 
            "problem": "Clasificación de agua potable", 
            "input": "ph, Hardness, Solids, Chloramines, Sulfate, Conductivity, Organic_carbon, Trihalomethanes, Turbidity", 
            "output": "potabilidad (0 o 1)"}


@app.post("/potabilidad/")
def predict_water_quality(data: WaterQuality):
    input_data = [[
        data.ph,
        data.Hardness,
        data.Solids,
        data.Chloramines,
        data.Sulfate,
        data.Conductivity,
        data.Organic_carbon,
        data.Trihalomethanes,
        data.Turbidity
    ]]
    
    prediction = model.predict(input_data)

    return {"potabilidad": int(prediction[0])}


# **3. Docker (2 puntos)**

<div align="center">
  <img src="https://miro.medium.com/v2/resize:fit:1400/1*9rafh2W0rbRJIKJzqYc8yA.gif" width="500">
</div>

Tras el éxito de su aplicación web para generar la salida, Smapina le solicita que genere un contenedor para poder ejecutarla en cualquier computador de la empresa de agua potable.

## **3.1 Creación de Container (1 punto)**

Cree un Dockerfile que use una imagen base de Python, copie los archivos del proyecto e instale las dependencias desde un `requirements.txt`. Con esto, construya y ejecute el contenedor Docker para la API configurada anteriormente. Entregue el código fuente (incluyendo `main.py`, `requirements.txt`, y `Dockerfile`) y la imagen Docker de la aplicación. Para la dockerización, asegúrese de cumplir con los siguientes puntos:

1. **Generar un archivo `.dockerignore`** que ignore carpetas y archivos innecesarios dentro del contenedor.
2. **Configurar un volumen** que permita la persistencia de los datos en una ruta local del computador.
3. **Exponer el puerto** para acceder a la ruta de la API sin tener que entrar al contenedor directamente.
4. **Incluir imágenes en el notebook** que muestren la ejecución del contenedor y los resultados obtenidos.
5. **Revisar y comentar los recursos utilizados por el contenedor**. Analice si los contenedores son livianos en términos de recursos.

## **3.2 Preguntas de Smapina (1 punto)**
Tras haber experimentado con Docker, Smapina desea profundizar más en el tema y decide realizarle las siguientes consultas:

- ¿Cómo se diferencia Docker de una máquina virtual (VM)?
- ¿Cuál es la diferencia entre usar Docker y ejecutar la aplicación directamente en el sistema local?
- ¿Cómo asegura Docker la consistencia entre diferentes entornos de desarrollo y producción?
- ¿Cómo se gestionan los volúmenes en Docker para la persistencia de datos?
- ¿Qué son Dockerfile y docker-compose.yml, y cuál es su propósito?

<div align="center">
  <img src="localhost800.png" width="500">
</div>

<div align="center">
  <img src="docker_stats.png" width="500">
</div>

El contenedor está utilizando recursos de manera eficiente, con un bajo consumo de CPU y memoria. Con un uso de solo el 0.24% de la CPU y el 5.12% de la memoria asignada, la aplicación dentro del contenedor no está llevando a cabo tareas intensivas que puedan afectar el rendimiento del sistema. Esto es positivo, ya que indica que el contenedor no está generando una carga excesiva en el host, lo que permite que otros procesos o contenedores puedan funcionar sin problemas de rendimiento.

Además, el uso de la red y el disco es mínimo, lo que sugiere que el contenedor no está gestionando grandes volúmenes de datos o realizando operaciones de I/O frecuentes. Este comportamiento es típico de aplicaciones ligeras que no requieren interacción intensiva con el sistema de archivos o la red. En general, los recursos utilizados por el contenedor son bajos, lo que refleja una implementación optimizada y eficiente que no presenta preocupaciones sobre el consumo de recursos.

*¿Cómo se diferencia Docker de una máquina virtual (VM)?*

Algunas de las principales diferencias es que las VMs virtualizan el hardware completo, incluyendo el sistema operativo, lo que las hace más pesadas en términos de recursos, ya que cada VM requiere su propio Sistema operativo invitado. Por otro lado, Docker utiliza contenedores que comparten el mismo kernel del SO anfitrión, lo que los hace más ligeros y eficientes. 

Otra diferencia clave es el nivel de abstracción que manejan. Las máquinas virtuales trabajan a nivel del hardware físico, lo que permite ejecutar varios sistemas operativos en un mismo servidor, pero con un mayor consumo de recursos. En cambio, Docker opera a nivel de las aplicaciones y sus entornos, haciendo que los contenedores sean más livianos y fáciles de trasladar entre distintos sistemas. Esto hace que Docker sea especialmente útil para proyectos que requieren agilidad y escalabilidad en el desarrollo, mientras que las máquinas virtuales se enfocan más en situaciones donde es necesario simular entornos completos o gestionar múltiples sistemas operativos.


*¿Cuál es la diferencia entre usar Docker y ejecutar la aplicación directamente en el sistema local?*

La principal diferencia entre usar Docker y ejecutar una aplicación directamente en el sistema local radica en el aislamiento y la portabilidad. Docker encapsula la aplicación junto con todas sus dependencias en un contenedor, garantizando que funcione de manera consistente sin importar el entorno donde se ejecute. Esto evita conflictos entre versiones de bibliotecas o configuraciones del sistema. En cambio, al ejecutar una aplicación directamente en el sistema local, ésta depende del entorno específico del sistema operativo, lo que puede generar problemas si otras aplicaciones requieren versiones diferentes de las mismas dependencias o si se cambia de máquina.

*¿Cómo asegura Docker la consistencia entre diferentes entornos de desarrollo y producción?*

Docker asegura la consistencia entre entornos de desarrollo y producción al encapsular las aplicaciones junto con todas sus dependencias, configuraciones y bibliotecas necesarias en contenedores. Estos contenedores funcionan de manera independiente del sistema operativo subyacente y garantizan que el comportamiento de la aplicación sea el mismo sin importar dónde se ejecute. Al usar imágenes Docker para construir los contenedores, los desarrolladores pueden definir con precisión el entorno requerido, eliminando problemas derivados de diferencias entre configuraciones locales y de producción, lo que mejora la predictibilidad y facilita la integración continua.

*¿Cómo se gestionan los volúmenes en Docker para la persistencia de datos?*


Los volúmenes en Docker son una forma de guardar datos que no se pierden aunque el contenedor se borre o reinicie. Funcionan como una carpeta especial en la computadora que Docker puede usar para almacenar información importante. Estos volúmenes se pueden compartir entre varios contenedores, lo que permite que todos accedan a los mismos datos si es necesario. Además, son fáciles de crear y usar con comandos simples, y se conectan al contenedor cuando lo inicias. Esto asegura que los datos estén seguros y siempre disponibles, sin importar lo que pase con los contenedores.


*¿Qué son Dockerfile y docker-compose.yml, y cuál es su propósito?*

Un Dockerfile es un archivo de texto donde se definen, paso a paso, las instrucciones para crear una imagen Docker. Es como una receta que incluye el sistema operativo base, las dependencias necesarias, las configuraciones y cómo se ejecutará la aplicación dentro del contenedor. Esto asegura que cualquiera pueda construir la misma imagen de manera consistente y reproducible.

El docker-compose.yml, por otro lado, es un archivo donde se define y orquesta la configuración de múltiples contenedores Docker. Permite especificar cómo se conectan entre sí, qué redes usan, qué volúmenes comparten y otros detalles necesarios para ejecutar aplicaciones complejas. Es ideal para proyectos que necesitan varios servicios trabajando juntos, como una base de datos, un servidor web y una aplicación.

# Conclusión
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por mail o U-cursos.

<div align="center">
  <img src="https://i.pinimg.com/originals/84/5d/f1/845df1aefc6a5e37ae575327a0cc6e43.gif" width="500">
</div>